[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module4/07-dashboards.ipynb)

# Interactive Dashboards: Plotly and Dash

**Module 4 — Data Science & Visualization** | Estimated time: 30 minutes

## Learning Objectives

By the end of this notebook you will be able to:
- Create interactive charts with Plotly Express (`px`)
- Build custom multi-trace figures with `go.Figure`
- Add hover data, animation frames, and 3D scatter plots
- Build a Dash application with layout components and callbacks
- Run Dash in a Colab environment
- Understand deployment options for sharing dashboards

In [ ]:
!pip install plotly dash jupyter-dash --quiet

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f'Plotly version: {px.__module__.split(".")[0]}')
import plotly
print(f'Plotly version: {plotly.__version__}')

rng = np.random.default_rng(42)

## 1. Generating a Rich Dataset

We create a synthetic global sales dataset that we will use for all chart examples.

In [ ]:
n = 500
countries = ['USA', 'Germany', 'Japan', 'Brazil', 'India', 'UK', 'France', 'Canada', 'Australia', 'Mexico']
latitudes  = {'USA': 38, 'Germany': 51, 'Japan': 36, 'Brazil': -15, 'India': 20,
               'UK': 54, 'France': 46, 'Canada': 56, 'Australia': -27, 'Mexico': 23}
longitudes = {'USA': -97, 'Germany': 10, 'Japan': 138, 'Brazil': -51, 'India': 77,
               'UK': -2, 'France': 2, 'Canada': -96, 'Australia': 134, 'Mexico': -102}

country_col  = rng.choice(countries, n)
category_col = rng.choice(['Electronics', 'Apparel', 'Food & Bev', 'Software'], n)
year_col     = rng.choice([2021, 2022, 2023, 2024], n)
revenue_col  = rng.exponential(500, n).round(2) + 100
units_col    = rng.integers(10, 300, n)
margin_col   = rng.uniform(0.1, 0.5, n).round(3)

df = pd.DataFrame({
    'country':  country_col,
    'lat':      [latitudes[c] for c in country_col],
    'lon':      [longitudes[c] for c in country_col],
    'category': category_col,
    'year':     year_col,
    'revenue':  revenue_col,
    'units':    units_col,
    'margin':   margin_col,
    'profit':   (revenue_col * margin_col).round(2)
})

print(f'Dataset shape: {df.shape}')
print(df.head())

## 2. Plotly Express: Core Charts

Plotly Express (`px`) provides one-line chart creation. Every chart returns a `go.Figure` object that can be further customized.

In [ ]:
# Scatter plot
fig = px.scatter(
    df, x='units', y='revenue', color='category', size='profit',
    hover_data=['country', 'year', 'margin'],
    title='Revenue vs Units Sold — colored by category, sized by profit',
    labels={'units': 'Units Sold', 'revenue': 'Revenue ($)'},
    template='plotly_white', width=850, height=450
)
fig.update_traces(marker=dict(opacity=0.7, line=dict(width=0.5, color='white')))
fig.show()

# Bar chart
df_country = df.groupby(['country', 'category'])['revenue'].sum().reset_index()
fig2 = px.bar(
    df_country, x='country', y='revenue', color='category', barmode='stack',
    title='Total Revenue by Country and Category',
    template='plotly_white', width=850, height=400
)
fig2.update_layout(xaxis_tickangle=-30)
fig2.show()

## 3. Animated Charts and Histogram / Box

In [ ]:
# Animated scatter: watch revenue distribution shift by year
df_anim = df.groupby(['year', 'country', 'category']).agg(
    revenue=('revenue', 'sum'), units=('units', 'sum'), profit=('profit', 'sum')
).reset_index()

fig_anim = px.scatter(
    df_anim, x='units', y='revenue',
    animation_frame='year',
    animation_group='country',
    color='category', size='profit',
    hover_name='country',
    title='Revenue vs Units (animated by year)',
    template='plotly_white', width=850, height=480,
    range_x=[0, df_anim['units'].max() * 1.1],
    range_y=[0, df_anim['revenue'].max() * 1.1]
)
fig_anim.show()

# Histogram
fig_hist = px.histogram(
    df, x='revenue', color='category', nbins=40, barmode='overlay',
    marginal='box',
    title='Revenue Distribution by Category',
    template='plotly_white', opacity=0.65, width=850, height=420
)
fig_hist.show()

# Box plot
fig_box = px.box(
    df, x='category', y='revenue', color='category',
    points='outliers',
    title='Revenue Distribution (Box)',
    template='plotly_white', width=750, height=400
)
fig_box.show()

## 4. Choropleth Map and 3D Scatter

In [ ]:
# Choropleth map: total revenue by country
df_map = df.groupby('country')['revenue'].sum().reset_index()

# ISO alpha-3 codes for choropleth
iso_map = {'USA': 'USA', 'Germany': 'DEU', 'Japan': 'JPN', 'Brazil': 'BRA',
           'India': 'IND', 'UK': 'GBR', 'France': 'FRA', 'Canada': 'CAN',
           'Australia': 'AUS', 'Mexico': 'MEX'}
df_map['iso'] = df_map['country'].map(iso_map)

fig_map = px.choropleth(
    df_map, locations='iso', color='revenue',
    hover_name='country',
    color_continuous_scale='Viridis',
    title='Total Revenue by Country',
    width=850, height=450
)
fig_map.show()

# 3D Scatter
fig_3d = px.scatter_3d(
    df, x='revenue', y='units', z='profit',
    color='category', opacity=0.6, size_max=8,
    title='3D Scatter: Revenue / Units / Profit',
    template='plotly_white', width=800, height=550
)
fig_3d.show()

## 5. go.Figure: Custom Multi-trace Layout

For full control, use `go.Figure` directly and add traces manually.

In [ ]:
monthly = df.copy()
monthly['month'] = rng.integers(1, 13, len(df))
monthly_rev = monthly.groupby(['year', 'month'])['revenue'].sum().reset_index()
monthly_rev = monthly_rev.sort_values(['year', 'month'])
monthly_rev['month_label'] = monthly_rev['month'].map(
    {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
     7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'})

colors = {2021: '#636EFA', 2022: '#EF553B', 2023: '#00CC96', 2024: '#AB63FA'}

fig_multi = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Monthly Revenue by Year', 'Revenue Share by Category']
)

for year in sorted(monthly_rev['year'].unique()):
    sub = monthly_rev[monthly_rev['year'] == year]
    fig_multi.add_trace(
        go.Scatter(x=sub['month_label'], y=sub['revenue'],
                   mode='lines+markers', name=str(year),
                   line=dict(color=colors[year], width=2)),
        row=1, col=1
    )

category_totals = df.groupby('category')['revenue'].sum()
fig_multi.add_trace(
    go.Pie(labels=category_totals.index, values=category_totals.values,
           hole=0.4, name='Category'),
    row=1, col=2
)

fig_multi.update_layout(
    title_text='Sales Dashboard — Multi-trace Layout',
    template='plotly_white', width=1000, height=450,
    legend=dict(orientation='v', x=0.42)
)
fig_multi.show()

## 6. Dash Application with Callbacks

**Dash** is a Python framework for building reactive web dashboards. It uses Flask under the hood. A Dash app has two parts: the **layout** (what components appear) and **callbacks** (what happens when the user interacts).

> In Colab, Dash needs to run with `jupyter-dash` or be exposed via a tunnel. The cell below defines the app. Run the final cell to launch it.

In [ ]:
from dash import Dash, dcc, html, Input, Output, callback
import plotly.express as px

app = Dash(__name__)

all_categories = df['category'].unique().tolist()
all_years = sorted(df['year'].unique().tolist())

app.layout = html.Div([
    html.H2('Sales Dashboard', style={'fontFamily': 'Arial', 'textAlign': 'center'}),

    html.Div([
        html.Div([
            html.Label('Category'),
            dcc.Dropdown(
                id='category-filter',
                options=[{'label': c, 'value': c} for c in all_categories],
                value=all_categories,
                multi=True,
                clearable=False
            )
        ], style={'width': '45%', 'display': 'inline-block', 'paddingRight': '20px'}),

        html.Div([
            html.Label('Year Range'),
            dcc.RangeSlider(
                id='year-slider',
                min=min(all_years), max=max(all_years), step=1,
                value=[min(all_years), max(all_years)],
                marks={y: str(y) for y in all_years}
            )
        ], style={'width': '45%', 'display': 'inline-block'})
    ], style={'padding': '15px', 'backgroundColor': '#f9f9f9', 'borderRadius': '8px', 'margin': '10px'}),

    html.Div([
        dcc.Graph(id='scatter-chart', style={'display': 'inline-block', 'width': '50%'}),
        dcc.Graph(id='bar-chart',     style={'display': 'inline-block', 'width': '50%'})
    ]),

    dcc.Graph(id='revenue-trend')
], style={'maxWidth': '1100px', 'margin': 'auto', 'fontFamily': 'Arial'})


@callback(
    Output('scatter-chart', 'figure'),
    Output('bar-chart', 'figure'),
    Output('revenue-trend', 'figure'),
    Input('category-filter', 'value'),
    Input('year-slider', 'value')
)
def update_charts(selected_categories, year_range):
    filtered = df[
        (df['category'].isin(selected_categories)) &
        (df['year'] >= year_range[0]) &
        (df['year'] <= year_range[1])
    ]

    # Scatter
    scatter = px.scatter(
        filtered, x='units', y='revenue', color='category',
        hover_data=['country', 'year'],
        title='Revenue vs Units',
        template='plotly_white'
    )

    # Bar
    bar_data = filtered.groupby('country')['revenue'].sum().reset_index()
    bar = px.bar(
        bar_data.sort_values('revenue', ascending=True).tail(8),
        x='revenue', y='country', orientation='h',
        title='Top Countries by Revenue',
        template='plotly_white', color='revenue',
        color_continuous_scale='Blues'
    )

    # Trend
    trend_data = filtered.groupby(['year', 'category'])['revenue'].sum().reset_index()
    trend = px.line(
        trend_data, x='year', y='revenue', color='category',
        markers=True, title='Revenue Trend by Year',
        template='plotly_white'
    )

    return scatter, bar, trend

print('Dash app defined.')
print('To run in Colab, execute: app.run(debug=False, port=8050)')
print('Then open the Colab port forwarding URL.')

## 7. Running the Dash App in Colab

In [ ]:
# Uncomment and run to launch the app in Colab.
# Colab will provide a link to the forwarded port.

# app.run(debug=False, port=8050, jupyter_mode='inline')

# --- Deployment Overview ---
print('Dash Deployment Options:')
print()
print('  1. Render.com        — Free tier, deploy from GitHub')
print('                         gunicorn server: app.server = app.server')
print()
print('  2. Railway.app       — Simple Git-push deployment')
print('                         Add Procfile: web: gunicorn app:server')
print()
print('  3. HuggingFace Spaces— Great for data science apps')
print('                         Set SDK=gradio or use docker')
print()
print('  4. PythonAnywhere    — Easy for beginners')
print()
print('  5. AWS/GCP/Azure     — Production-grade, more setup required')
print()
print('Requirements file needed:')
print('  dash>=2.0')
print('  plotly>=5.0')
print('  pandas')
print('  gunicorn  # for production server')

## Practice Exercises

**Exercise 1 — Custom go.Figure**
Using the dataset, create a `go.Figure` that overlays a bar chart (category totals) and a scatter plot (individual data points) on the same axes. Add a `go.Layout` with a custom title, axis labels, and hover mode set to `'x unified'`.

**Exercise 2 — Callback Chain**
Extend the Dash app with a third Input: a `dcc.Checklist` for selecting regions (North, South, East, West if you add a region column). Chain it into the existing callback so all three filters apply simultaneously.

**Exercise 3 — Animated Choropleth**
Using `px.choropleth`, create an animated world map that shows total revenue by country for each year (2021-2024) using `animation_frame='year'`. Add a custom color scale and a descriptive title.